In [7]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler

In [8]:
loan_data = pd.read_csv('Loan_Default.csv')
loan_data.drop(['ID', 'year'], axis=1, inplace =True)

In [9]:
categorical_features = loan_data.select_dtypes(include=['object']).columns.tolist()
high_missing_cols = ['rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'property_value', 'LTV', 'dtir1']
Cols_to_be_imputed = ['term', 'income', 'age', 'loan_limit_freq', 'approv_in_adv_freq', 'loan_purpose_freq', 'Neg_ammortization_freq', 'submission_of_application_freq']
loan_data =  loan_data.drop(high_missing_cols, axis=1)

C:\Users\mobig\AppData\Local\Temp\ipykernel_30704\2732357333.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = loan_data.select_dtypes(include=['object']).columns.tolist()


In [10]:
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

In [11]:
enc = OrdinalEncoder()
loan_data[Ordinal_features] = enc.fit_transform(loan_data[Ordinal_features])

In [12]:
for c in Nominal_features:
    
    frequency = loan_data[c].value_counts(normalize=True)
    loan_data[c+'_freq'] = loan_data[c].map(frequency)

loan_data =  loan_data.drop(Nominal_features, axis=1)

In [13]:
for c in (Cols_to_be_imputed):
    
    loan_data[c].fillna(loan_data[c].mean(), inplace = True)

C:\Users\mobig\AppData\Local\Temp\ipykernel_30704\812272468.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  loan_data[c].fillna(loan_data[c].mean(), inplace = True)


### Feature Selection (Filter Methods)

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import mutual_info_classif

In [15]:
y = loan_data['loan_amount']
X = loan_data.drop(['loan_amount'], axis = 1)

X_train, X_test, y_train, y_test = train_test_split(X, y,random_state=100, test_size=0.3)

### Information Gain

In [16]:
X_train1 = X_train.copy()
X_test1 = X_test.copy()

In [17]:
# It measures how much information a feature gives us about the target.
# Higher value → more useful feature
# Mutual information values are not percentages and have no fixed maximum.
mutual_info = mutual_info_classif(X_train1, y_train)
mutual_info

ValueError: Input X contains NaN.

In [ ]:
mutual_info = pd.Series(mutual_info)
mutual_info.index = X_train1.columns
mutual_info = mutual_info.sort_values(ascending=False)
mutual_info.plot.bar(figsize=(20, 8))

### Correlation Cofficient

In [ ]:
X_train2 = X_train.copy()
X_test2 = X_test.copy()

In [ ]:
cor = X_train2.corr()
plt.figure(figsize=(20,10))
sns.heatmap(cor, cmap=plt.cm.CMRmap_r,annot=True)
plt.show() 

In [ ]:
corr_matrix = X_train.corr().abs()
threshold = 0.7

corr_features = set()

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > threshold:
            colname = corr_matrix.columns[i]
            corr_features.add(colname)
        
corr_features

In [ ]:
#Removing correlated features
X_train2.drop(corr_features,axis=1)
X_test2.drop(corr_features,axis=1)

### Variance Threshold

In [ ]:
X_train3 = X_train.copy()
X_test3 = X_test.copy()

In [ ]:
th = 0.001
var_thres=VarianceThreshold(threshold=th)
var_thres.fit(X_train3 )
new_cols = var_thres.get_support()
new_cols
# X_train3.iloc[:,new_cols]

### Feature Selection (Wrapper Methods)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import ExtraTreesClassifier
from mlxtend.feature_selection import SequentialFeatureSelector as sfs

### Forward Feature Selection

In [ ]:
X_train4 = X_train.copy()
X_test4 = X_test.copy()
clf = ExtraTreesClassifier(n_estimators=60, n_jobs=-1,random_state=42) #It creates an Extra Trees classifier with 60 trees, using all CPU cores, and fixed randomness for reproducibility.
len(X_train4.columns)

In [ ]:
# Build step forward feature selection
sfs1 = sfs(clf,
           k_features=10, # to automatically find best number of features use"best"
           forward=True,
           floating=False, #No removal of features once added
           verbose=1, #Show progress while selecting
           scoring='accuracy',
           cv=9 #Use 9-fold cross-validation to test performance
          )

new_X_train = sfs1.fit(X_train4, y_train)

In [ ]:
sfs1.k_feature_names_

### Recursive Feature Elimination

In [ ]:
rfe = RFE(clf, n_features_to_select=10)
new_X_train = rfe.fit(X_train4, y_train) 
print(len(rfe.support_))
print(rfe.support_)
X_train4.columns[rfe.support_]

### Dimensionality Reduction

In [ ]:
sc = StandardScaler()
loan_data = pd.DataFrame(sc.fit_transform(loan_data), columns=loan_data.columns)  #fit and transforming StandardScaler the dataframe

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=10)
trans_loan_data = pca.fit_transform(loan_data)
trans_loan_data

In [ ]:
pca.n_components_

In [ ]:
pca.explained_variance_

In [ ]:
features = range(pca.n_components_)
plt.bar(features, pca.explained_variance_)
plt.xlabel('PCA feature')
plt.ylabel('Variance')
plt.xticks(features)
plt.show()